In [1]:
#Cell 1:importing modules
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm 
from collections import Counter # For building vocabulary
import sys 

In [2]:
#Step 1: Uploading and Clearing Database
csv_file_path = r'C:\Users\Kutay\Documents\jupyter\csv\topic_classification_data.csv' 
text_column_name = 'clean_content' 
label_column_name = 'label'         

print(f"Loading data from: {csv_file_path}")
df = pd.read_csv(csv_file_path)
print("CSV file loaded successfully!")

initial_rows = len(df)
df.dropna(subset=[text_column_name, label_column_name], inplace=True)
if len(df) < initial_rows:
     print(f"\nRemoved {initial_rows - len(df)} rows with missing text or labels.")

df[text_column_name] = df[text_column_name].astype(str)


Loading data from: C:\Users\Kutay\Documents\jupyter\csv\topic_classification_data.csv
CSV file loaded successfully!

Removed 100 rows with missing text or labels.


In [3]:
def remove_urls(text):
  text_str = str(text) 
  url_pattern = r'https?://\S+|http\s+\S+'
  cleaned_text = re.sub(url_pattern, '', text_str)
  cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
  return cleaned_text

print("URL removal function defined.")

URL removal function defined.


In [4]:
print("Applying URL removal...")
cleaned_texts = df[text_column_name].apply(remove_urls).tolist()
print(f"Applied URL removal to {len(cleaned_texts)} texts.")

original_labels = df[label_column_name].values
unique_labels = sorted(list(set(original_labels)))
num_classes = len(unique_labels)
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()} 

labels_numeric = [label_to_id[label] for label in original_labels]

print(f"\nFound {num_classes} unique classes.")
print("Label to ID mapping:", label_to_id)

Applying URL removal...
Applied URL removal to 136024 texts.

Found 6 unique classes.
Label to ID mapping: {'Emotion': 0, 'Financial': 1, 'Health': 2, 'Politics': 3, 'Science': 4, 'Sport': 5}


In [5]:
#Step 2:Splitting the Data
print("\nSplitting data (80% train / 20% test)...")
X_train_text, X_test_text, y_train_numeric, y_test_numeric = train_test_split(
    cleaned_texts,
    labels_numeric, 
    test_size=0.2,
    random_state=73, 
    stratify=labels_numeric 
)
print(f" - Training samples: {len(X_train_text)}")
print(f" - Testing samples: {len(X_test_text)}")


Splitting data (80% train / 20% test)...
 - Training samples: 108819
 - Testing samples: 27205


In [6]:
#Vocab Creating 
print("\nBuilding vocabulary from training data...")
word_counts = Counter()
for text in tqdm(X_train_text, desc="Counting words"):
    word_counts.update(text.lower().split())


MIN_FREQ = 3 
vocab = [word for word, count in word_counts.items() if count >= MIN_FREQ]
vocab = sorted(vocab)

vocab = sorted(word_counts, key=word_counts.get, reverse=True)

word2idx = {word: i+2 for i, word in enumerate(vocab)} 
word2idx['<PAD>'] = 0 
word2idx['<UNK>'] = 1 
vocab_size = len(word2idx)

print(f"Vocabulary Size: {vocab_size}")


Building vocabulary from training data...


Counting words: 100%|██████████████████████████████████████████████████████| 108819/108819 [00:00<00:00, 166289.57it/s]

Vocabulary Size: 97549


In [7]:
#Creating our dataset class
class TopicDataset(Dataset):
    def __init__(self, texts, labels_numeric, word2idx, max_len):
        self.texts = texts
        self.labels = labels_numeric
        self.word2idx = word2idx
        self.max_len = max_len
        self.pad_idx = word2idx['<PAD>']
        self.unk_idx = word2idx['<UNK>']

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = str(self.texts[idx]).lower().split()
        indices = [self.word2idx.get(word, self.unk_idx) for word in words]

        if len(indices) < self.max_len:
            indices += [self.pad_idx] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]

        label = self.labels[idx]

        return torch.tensor(indices), torch.tensor(label, dtype=torch.long)

print("TopicDataset class defined.")

TopicDataset class defined.


In [8]:
#Creating the BiLSTM Architecture
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.5, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes) 
        self.dropout = nn.Dropout(dropout)

    def forward(self, text_indices):
        embedded = self.dropout(self.embedding(text_indices))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        hidden_cat = torch.cat([hidden[-2,:,:], hidden[-1,:,:]], dim=1)

        return self.fc(self.dropout(hidden_cat)) 

print("LSTMClassifier class defined.")

LSTMClassifier class defined.


In [9]:
#Hyperparameters for fine tuning and seperating the texts and labels for training.
MAX_LEN = 100       # Max sequence length
BATCH_SIZE = 64     # Sentence batch size in one go
EMBED_DIM = 200    
HIDDEN_DIM = 384    
NUM_LAYERS = 1      # LSTM layers
DROPOUT = 0.5
LEARNING_RATE = 0.001
NUM_EPOCHS = 20      

print("\nCreating Datasets...")
train_dataset = TopicDataset(X_train_text, y_train_numeric, word2idx, max_len=MAX_LEN)
test_dataset = TopicDataset(X_test_text, y_test_numeric, word2idx, max_len=MAX_LEN)

print("Creating DataLoaders...")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

print("Initializing model...")
model = LSTMClassifier(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes, 
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    padding_idx=word2idx['<PAD>']
).to(device)

print(f"\nModel architecture:\n{model}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
# Optional: Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)


Creating Datasets...
Creating DataLoaders...

Using device: cuda
Initializing model...

Model architecture:
LSTMClassifier(
  (embedding): Embedding(97549, 200, padding_idx=0)
  (lstm): LSTM(200, 384, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=768, out_features=6, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

Total parameters: 21,314,606


In [10]:
#Training and evalution functions defining.
def train_epoch_lstm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for texts_batch, labels_batch in tqdm(loader, desc="Training", leave=False):
        texts_batch = texts_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(texts_batch)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

def evaluate_lstm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for texts_batch, labels_batch in tqdm(loader, desc="Evaluating", leave=False):
            texts_batch = texts_batch.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(texts_batch)
            loss = criterion(outputs, labels_batch)

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

print("Training and evaluation functions defined.")

Training and evaluation functions defined.


In [11]:
#TRAINING
print("\nStarting Training...")
best_test_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

    train_loss, train_acc = train_epoch_lstm(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate_lstm(model, test_loader, criterion, device)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc*100:.2f}%")

    # Optional: Adjust learning rate with scheduler based on test accuracy
    scheduler.step(test_acc)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_bilstm_model.pth')
        print(f"  ✨ New best model saved with Test Acc: {best_test_acc*100:.2f}%")

print(f"\n{'='*50}")
print(f"Training Complete!")
print(f"Best Test Accuracy achieved: {best_test_acc*100:.2f}%")
print(f"{'='*50}\n")


Starting Training...

--- Epoch 1/20 ---


  Train Loss: 0.7583 | Train Acc: 73.56%
  Test Loss:  0.5517 | Test Acc:  81.90%
  ✨ New best model saved with Test Acc: 81.90%

--- Epoch 2/20 ---


  Train Loss: 0.4746 | Train Acc: 84.33%
  Test Loss:  0.4043 | Test Acc:  87.08%
  ✨ New best model saved with Test Acc: 87.08%

--- Epoch 3/20 ---


  Train Loss: 0.3921 | Train Acc: 87.04%
  Test Loss:  0.3669 | Test Acc:  88.53%
  ✨ New best model saved with Test Acc: 88.53%

--- Epoch 4/20 ---


  Train Loss: 0.3431 | Train Acc: 88.65%
  Test Loss:  0.3529 | Test Acc:  88.78%
  ✨ New best model saved with Test Acc: 88.78%

--- Epoch 5/20 ---


...

  Train Loss: 0.1525 | Train Acc: 94.83%
  Test Loss:  0.3772 | Test Acc:  90.33%

--- Epoch 17/20 ---


  Train Loss: 0.1488 | Train Acc: 94.94%
  Test Loss:  0.3761 | Test Acc:  90.36%

--- Epoch 18/20 ---


  Train Loss: 0.1469 | Train Acc: 95.03%
  Test Loss:  0.3752 | Test Acc:  90.45%
  ✨ New best model saved with Test Acc: 90.45%

--- Epoch 19/20 ---


  Train Loss: 0.1462 | Train Acc: 95.09%
  Test Loss:  0.3752 | Test Acc:  90.43%

--- Epoch 20/20 ---


  Train Loss: 0.1422 | Train Acc: 95.21%
  Test Loss:  0.3810 | Test Acc:  90.44%

Training Complete!
Best Test Accuracy achieved: 90.45%



In [12]:
#Getting the results.
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

best_model = LSTMClassifier(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    padding_idx=word2idx['<PAD>']
).to(device)


best_model.load_state_dict(torch.load('best_bilstm_model.pth', map_location=device))
print("Successfully loaded best_bilstm_model.pth")

best_model.eval()

all_predictions = []
all_labels = []

print("\nEvaluating the best model on the test set...")
with torch.no_grad():
    for texts_batch, labels_batch in tqdm(test_loader, desc="Testing"):
        texts_batch = texts_batch.to(device)
        labels_batch = labels_batch.to(device) 

        outputs = best_model(texts_batch)
        _, predicted = torch.max(outputs.data, 1) 

        all_predictions.extend(predicted.cpu().numpy()) #
        all_labels.extend(labels_batch.cpu().numpy())   

print("\n--- Final Evaluation Results ---")
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Overall Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
target_names = [id_to_label[i] for i in range(num_classes)]
print(classification_report(all_labels, all_predictions, target_names=target_names))

print("--------------------------------")

Successfully loaded best_bilstm_model.pth

Evaluating the best model on the test set...


Testing: 100%|███████████████████████████████████████████████████████████████████████| 426/426 [00:04<00:00, 94.06it/s]


--- Final Evaluation Results ---
Overall Test Accuracy: 0.9045 (90.45%)

Classification Report:
              precision    recall  f1-score   support

     Emotion       0.91      0.96      0.93      5652
   Financial       0.88      0.87      0.88      4596
      Health       0.91      0.90      0.91      7265
    Politics       0.93      0.93      0.93      7555
     Science       0.68      0.56      0.61       922
       Sport       0.89      0.84      0.87      1215

    accuracy                           0.90     27205
   macro avg       0.87      0.85      0.86     27205
weighted avg       0.90      0.90      0.90     27205

--------------------------------
